# TP9- Exploitation and Exploration

Estimated time: **1.5** hour


In this lab, you'll go over hands-on exercises using Python to further consolidate your understanding about balancing exploitation and exploration in learning algorithms, illustrated with the multi-armed bandit problem. We will explore and compare different strategies, including the Random strategy, the Epsilon-Greedy strategy with or without decay, the Optimistic Initial Value (OIV) strategy.

## Objectives


* Get familiar with the dataset
* Sample exercises
  * Introduction of the multi-armed bandit problem
  * Random strategy
  * Epsilon-Greedy strategy
  * Epsilon-Greedy strategy with decaying epsilon
  * Optimistic Initial Value (OIV) strategy
* Assignment questions

----


### Import Libraries


In [ ]:
import pandas as pd
import numpy as np 
import random
import matplotlib.pyplot as plt

# The multi-armed bandit problem at a glance

The **multi-armed bandit problem** is a classic framework in decision theory and learning algorithms that illustrates the **exploration–exploitation trade-off**.

### 🎯 The problem
A gambler in a casino has access to several slot machines and wants to **maximize his gains over a series of pulls**.

### ❓ The challenge
**Different slot machines have different winning probabilities**. Some machines might have higher chances of giving rewards than others, but the gambler does not know these probabilities in advance.

### 🧠 Strategies we will compare
Different strategies can be used to address the multi-armed bandit problem:

>**Random strategy** → **Epsilon-greedy strategy** → **Optimistic initial values strategy**

In the following, we will explore and compare these different strategies.

# Example exercises

## <span style="color:blue"> Emulate 10 slot machines, each with a randomly generated winning probability

In [ ]:
num_arms = 10

# Generate random winning probabilities for the 10 slot machines
np.random.seed(42)
reward_probs = np.random.uniform(0, 1, num_arms)

print(reward_probs.round(2))

Above we synthesize 10 slot machines (a.k.a 10 arms), **each with its own winning probability**.

## <span style="color:blue"> Visualize the winning probabilities for the 10 slot machines

In [ ]:
# Compute the average probability and the best reward probability
average_prob = np.mean(reward_probs)
best_reward_prob = np.max(reward_probs)
best_arm = np.argmax(reward_probs) 

# Print out the average and best reward probabilities
print(f"Average winning probability: {average_prob:.2f}")
print(f"Arm with the highest winning probability: {best_reward_prob:.2f} (Arm {best_arm})")


plt.figure(figsize=(8, 5))

# Plot the winning probabilities
arm_labels = [f"{i}" for i in range(num_arms)]
barlist = plt.bar(arm_labels, reward_probs, color='blue')

# Annotate each bar with its winning probability
for i, bar in enumerate(barlist):
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
             f"{reward_probs[i]:.2f}", ha='center', va='bottom', fontsize=10)

# Add labels and title
plt.xlabel('Arm Index', fontsize=12)
plt.ylabel('Winning Probability', fontsize=12)
plt.title('Winning probabilities of the 10 individual slot machines', fontsize=14)
plt.tight_layout()

# Show the plot
plt.show()

In the above code, we visualize the winning probability of each slot machine (a.k.a Arm) in a bar plot. We also print out the average winning probability,  as well as the slot machine with the highest winning probability. 

---

❓ <span style="color:purple;">**Quick test**</span>: **Suppose the winning probabilities of all 10 slot machines are known in advance and each win gives the same reward. Which slot machine should you keep playing to maximize your expected total reward?**

A. Arm #0, because it is the first slot machine

B. Arm #1, because it has the highest winning probability

C. Arm #7, because it also has a high winning probability

D. Select an arm randomly on every pull

<details>
<summary>Click to reveal the answer</summary>

**Answer**:

<p style="color:blue;">
<b>B. Arm #1, because it has the highest winning probability.</b>
</p>

**Explanation**:

<p style="color:blue;">
Arm #1 has the highest winning probability, at 0.95. Therefore, repeatedly selecting Arm #1 maximizes the expected total reward.
</p>

<p style="color:blue;">
Because the winning probabilities are already known, exploration is unnecessary. The gambler can focus entirely on exploitation by always selecting the best arm.
</p>

</details>

---

<div style="border-left: 5px solid #2563eb; background-color: #eff6ff;
            padding: 10px 14px; margin: 12px 0; border-radius: 4px;">

<strong>💡 Why is a learning algorithm needed?</strong>

<p style="margin-bottom: 0;">
Since the winning probabilities are <strong>unknown</strong>, the algorithm must balance
<strong>exploration</strong> and <strong>exploitation</strong> to identify the best slot machine
and <strong>maximize the total reward</strong>.
</p>

</div>

## <span style="color:blue"> Define a function to emulate pulling an arm and collecting the reward

In [ ]:
def pull_arm_and_get_reward(winning_probability):
    """Simulates the reward for a slot machine given its winning probability."""
    return 1 if np.random.random() < winning_probability else 0

### For simplicity, we assume that pulling an arm results in either a reward (1) or no reward (0). 

### With <small>`np.random.random()`</small> we generate a random float in the range [0.0, 1.0). And every number in this range is equally likely to occur (uniform distribution). 

### In one trail, if the random number generated is less than the probability, the trail is considered a success and the reward is 1. Otherwise, the reward is 0. 

### If we run a large number of trails, the average reward will converge to the winning probability of the corresponding arm. For example, if the winning probability is 0.7, any random number in the range [0.0, 0.7) will result in a reward of 1, and any random number in the range of [0.7, 1.0) will result in no reward. 

### This provides a convenient way to learn the winning probability of an arm.

# Example 1: Random strategy (exploration only)

## <span style="color:blue">Example 1.1: Run an experiment with 1000 trails, at each trail we pull a random arm

In [ ]:
def random_strategy(reward_probs, num_trials):
    """Implement the random strategy."""
    rewards = []
    selected_arms = []
    
    for _ in range(num_trials):
        # Select a random arm
        arm = np.random.choice(len(reward_probs))  
        
        # Pull the arm and obtain the reward, which depends on the winning 
        # probability of the selected arm 
        reward = pull_arm_and_get_reward(reward_probs[arm])
        
        # Keep track of the obtained reward and selected arm
        rewards.append(reward)
        selected_arms.append(arm)  
        
    return rewards, selected_arms


num_trials = 1000 

# Run an experiment with 1000 trails using the random strategy
random_rewards, random_selected_arms = random_strategy(reward_probs, num_trials)

### In the above code, we define a function <small>`random_strategy()`</small>, which runs an experiment wtih 1000 trails. In each trail, among the 10 arms, we pull an arm randomly and then collect the reward. The reward depends on the winning probability of the selected arm, it could be either 0 or 1. We also use two lists, <small>`selected_arms`</small> and <small>`rewards`</small>, to keep track of the selected arm and collected reward in each trail. 

## <span style="color:blue"> Example 1.2: Define a function to evaluate the performance of different strategies

In [ ]:
def compute_cumulative_average(rewards):
    """Computes the cumulative average of rewards."""
    return np.cumsum(rewards) / np.arange(1, len(rewards) + 1)
    
# Compute the cumulative average reward
random_avg_rewards = compute_cumulative_average(random_rewards)

### To compare the performance of the different strategies, we define a function <small>`compute_cumulative_average()`</small>, which computes the cumulative average reward obtained at the end of each trail. It is calculated as $\frac{\text{Sum of reward obtained up to trail k}}{\text{Number for trails up to trail k}}$.

### For instance, if the rewards collected after 5 trails are [1, 1, 1, 1, 0], then the cumulative average reward after each trail is: $\frac{1}{1}$ after the 1st trail, $\frac{1+1}{2}$ after the 2nd trail, $\frac{1+1+1}{3}$ after the 3rd trail, $\frac{1+1+1+1}{4}$ after the 4th trail, $\frac{1+1+1+1+0}{5}$ after the 5th trail. Hence, the cumulative average reward is [1.0, 1.0, 1.0, 1.0, 0.8].

### If a strategy is able to learn the arm with the highest winning probability with time by balancing exploitation and exploration, the cumulative average reward should also increase with time. 

In [ ]:
# Create a data frame which stores the trail index, the selected 
# arm, the obtained reward, and the cumulative average reward.
df = pd.DataFrame({
    "Trial Index": np.arange(len(random_rewards)),
    "Selected arm": random_selected_arms,
    "Reward Obtained": random_rewards,
    "Cumulative Average Reward": random_avg_rewards
})

# Display the first few rows of the data frame
df.head(10)  

### Above we display the selected arm, the obtained reward and the cumulative average reward at each trail for the random strategy

## <span style="color:blue"> Example 1.3: Visualize the selected arm and the cumulative average reward 

In [ ]:
plt.figure(figsize=(12, 6))

# Plot arm selection for each trail
plt.subplot(2, 1, 1)

plt.scatter(range(num_trials), random_selected_arms, alpha=0.5, color='blue', s=10)
plt.xlabel('Trial', fontsize=12)
plt.ylabel('Selected Arm', fontsize=12)
plt.title('Arm Selection in Random Strategy', fontsize=14)
# Ensure y-axis ticks correspond to arm indices
plt.yticks(range(num_arms))  
plt.grid(alpha=0.5)

# Plot cumulative average rewards
plt.subplot(2, 1, 2)

plt.plot(random_avg_rewards, color='orange', label='Random Strategy')
plt.axhline(np.mean(reward_probs), color='green', linestyle='--', label='Average Winning Probability')
plt.xlabel('Trial', fontsize=12)
plt.ylabel('Cumulative Average Reward', fontsize=12)
plt.title('Cumulative Average Reward obtained with Random Strategy', fontsize=14)
plt.ylim(0, 1)
plt.legend(fontsize=12)
plt.grid(alpha=0.5)

plt.tight_layout()
plt.show()

### The upper plot visualizes the arm selected at each trail. Each blue dot indicates that the corresponding arm is selected in a certain trail. Since at each trail the arm is randomly selected, as we can see, the distribution is random and the occurrence of each arm is quite similar to each other.

### The lower plot visualizes the cumulative average reward at the end of each trail. In addition, it also plots a horizontal line corresponding to the average winning probability of all the arms, which is computed as <small>`np.mean(reward_probs)`</small>. As we can see, the cumulative average reward converges towards the average winning probability, since we use a random strategy.

# Example 2: Epsilon greedy strategy

<div style="text-align: left;">
  <img src="Epsilon-Greedy.png" width="700">
</div>

<div style="border-left: 5px solid #0ea5e9; background-color: #f0f9ff;
            padding: 10px 14px; margin: 12px 0; border-radius: 4px;">

<p style="margin-bottom: 0;">
The slide presents the main algorithmic steps as <strong>pseudocode</strong>.
The implementation below adapts these steps to the variables, helper functions,
and required outputs used in this notebook.
</p>

</div>

## <span style="color:blue"> Example 2.1: Implement the epsilon greedy algorithm

In [ ]:
def epsilon_greedy_strategy(reward_probs, num_trials, epsilon):
    """Implements the epsilon-greedy strategy"""
    num_arms = len(reward_probs)
    rewards = []
    selected_arms = []  
    
    arm_counts = np.zeros(num_arms)  # Track the number of times each arm is pulled
    average_rewards = np.zeros(num_arms)  # Track the average reward for each arm

    # Sets the random seed for reproducibility
    np.random.seed(39)
    for _ in range(num_trials):
        """ Epsilon-greedy for each trail """
        if np.random.random() < epsilon:
            # Exploration: choose a random arm
            arm = np.random.choice(num_arms)
        else:
            # Exploitation: choose the best arm with the highest average reward so far
            arm = np.argmax(average_rewards)
        
        # Pull the chosen arm and get the reward
        reward = pull_arm_and_get_reward(reward_probs[arm])

        # Keep track of the obtained reward and selected arm
        rewards.append(reward)
        selected_arms.append(arm)  
        
        # Update the average reward for the selected arm
        k = arm_counts[arm]
        arm_counts[arm] += 1
        average_rewards[arm] = (average_rewards[arm]*k + reward) / arm_counts[arm]
    
    return rewards, selected_arms

### In the above code, at each trail we use the Epsilon-Greedy Strategy to determine the arm to be pulled: if the generated random number is less than <small>`epsilon`</small>, we perform exploration by selecting a random arm. Otherwise, we exploit the best action learned so far, that is selecting the arm with the highest average reward till this trail.

### To find the arm with the highest average reward, we keep track of the number of the times an arm is selected in <small>`arm_counts[arm]`</small> and also update the average reward every time an arm is selected in<small>`average_rewards[arm]`</small>.

## <span style="color:blue"> Example 2.2: Run an experiment with 1000 trails, at each trail the epsilon greedy strategy is used with epsilon set to 0.2

In [ ]:
epsilon = 0.2

# Run the epsilon-greedy strategy
epsilon_greedy_rewards, epsilon_greedy_selected_arms = epsilon_greedy_strategy(reward_probs, num_trials, epsilon)

# Compute cumulative average rewards
epsilon_greedy_avg_rewards = compute_cumulative_average(epsilon_greedy_rewards)

### We run the experiment again with 1000 trails, in each trail we select an arm using epsilon greedy strategy, with epsilon set to 0.2 ($ε = 0.2$). As before, we return the rewards collected in each trail as well as the selected arm. We also compute the cumulative average rewards for visualization next.

## <span style="color:blue"> Example 2.3: Visualize the selected arm and the cumulative average reward 

In [ ]:
# Visualization
plt.figure(figsize=(12, 6))

# Plot arm selection for each trail
plt.subplot(2, 1, 1)
plt.scatter(range(num_trials), epsilon_greedy_selected_arms, alpha=0.5, color='blue', s=10)
plt.xlabel('Trial', fontsize=12)
plt.ylabel('Selected Arm', fontsize=12)
plt.title(f'Arm Selection in Epsilon-Greedy Strategy (ε = {epsilon})', fontsize=14)
plt.yticks(range(num_arms))  # Ensure y-axis ticks correspond to arm indices
plt.grid(alpha=0.5)

# Plot cumulative average rewards
plt.subplot(2, 1, 2)
plt.plot(epsilon_greedy_avg_rewards, color='orange', label='Epsilon-Greedy Strategy')
plt.axhline(np.max(reward_probs), color='green', linestyle='--', label='Best Possible Reward')
plt.xlabel('Trial', fontsize=12)
plt.ylabel('Cumulative Average Reward', fontsize=12)
plt.title('Cumulative Average Reward in Epsilon-Greedy Strategy', fontsize=14)
plt.ylim(0, 1)
plt.legend(fontsize=12)
plt.grid(alpha=0.5)

plt.tight_layout()
plt.show()

### As we can see in the upper plot, most of the time it chooses the best-known option (arm with the best average reward) and with much lower probability it explores randomly other options, namely other arms. 

### It is worth noting that, a local optimal is identified at the beginning, namely arm 7 (winning probability: 0.87). However, epsilon greedy manages to get out of the local optimal by taking advantage of exploration and find the arm with the highest winning probability, which is arm 1 (winning probability: 0.95).

### In the lower plot, which visualizes the cumulative average reward, we plot also a horizontal line corresponding to the highest winning probability (Arm 1: 0.95). 

### As we can see from the lower plot, the cumulative average reward converges towards the highest winning probability.

## <span style="color:blue"> Example 2.4: Compare the result (cumulative average reward) with the one obtained with random strategy

In [ ]:
# Visualization
plt.figure(figsize=(12, 6))

# plot the cumulative average reward obtained with the random strategy
plt.plot(random_avg_rewards, label='Random Strategy', color='blue')

# plot the cumulative average reward obtained with the Epsilon Greedy
plt.plot(epsilon_greedy_avg_rewards, label=f'Epsilon-Greedy Strategy (ε = {epsilon})', color='orange')

# Plot the best possible reward, that is if we always pull the arm with highest winning probability
plt.axhline(np.max(reward_probs), color='green', linestyle='--', label='Best Possible Reward')

plt.xlabel('Trial', fontsize=12)
plt.ylabel('Average Reward', fontsize=12)
plt.title('Comparison of Random and Epsilon-Greedy Strategies (Cumulative Average Reward)', fontsize=14)
plt.legend(fontsize=12)
plt.ylim(0, 1)
plt.grid(alpha=0.5)
plt.show()

### The above plot visualizes the cumulative average reward obtained with random strategy and epsilon greedy in the same figure. 

### While random strategy converges towards the average winning probability of all the arms, epsilon greedy strategy converges towards the best possible reward, which corresponds to the case that we keep pulling the arm with the highest winning probability. 

### It shows that epsilon greedy strategy can learn better than random strategy by balancing exploitation and exploration. 

# Example 3: compare the impact of different epsilon values 

## <span style="color:blue"> Example 3.1: Try different epsilon values: $0.0, 0.2$ and $1.0$

In [ ]:
epsilon_values = [0.0, 0.2, 1.0]  # Different epsilon values to compare

# Run experiments for each epsilon and store results
results = {}
for epsilon in epsilon_values:
    rewards, selected_arms = epsilon_greedy_strategy(reward_probs, num_trials, epsilon)
    cum_avg_rewards = compute_cumulative_average(rewards)
    results[epsilon] = (cum_avg_rewards, selected_arms)

### In the above code, we run the epsilon greedy strategy with different epsilon values: 0.0, 0.2 and 1.0. 

### According to the Epsilon-Greedy Strategy quoted below, when $ε = 1.0$, we will always choose a random arm, hence there is only exploration. When $ε = 0.2$, we balance exploration and exploitation. When $ε = 0.0$, there is no exploration, only exploitation. 

        if np.random.random() < epsilon:
            arm = np.random.choice(num_arms)
        else:
            arm = np.argmax(average_rewards)

## <span style="color:blue">Example 3.2: Visualize the selected arm and cumulative average reward for each epsilon value

In [ ]:
# Plot cumulative average rewards for each epsilon
plt.figure(figsize=(12, 6))

for i, epsilon in enumerate(epsilon_values):
    cum_avg_rewards, selected_arms = results[epsilon]
    plt.plot(cum_avg_rewards, label=f"Epsilon = {epsilon}")
    
plt.axhline(np.max(reward_probs), color='green', linestyle='--', label='Best Possible Reward')
plt.xlabel("Trials")
plt.ylabel("Cumulative Average Reward")
plt.title("Cumulative Average Reward for Different Epsilon Values")
plt.legend()
plt.grid()
plt.show()

# Plot arm selection for each epsilon 
plt.figure(figsize=(10, 9))

for i, epsilon in enumerate(epsilon_values):
    avg_rewards, selected_arms = results[epsilon]
    plt.subplot(3, 1, i+1)
    plt.scatter(range(num_trials), selected_arms, alpha=0.5, color='blue', s=10)
    plt.xlabel('Trial', fontsize=12)
    plt.ylabel('Selected Arm', fontsize=12)
    plt.title(f'Arm Selection in Epsilon-Greedy Strategy (ε = {epsilon})', fontsize=14)
    plt.yticks(range(num_arms))  # Ensure y-axis ticks correspond to arm indices
    plt.grid(alpha=0.5)

plt.tight_layout()
plt.show()

### As we can see from the upper most plot corresponding to the obtained cumulative average reward, $ε = 0.2$ outperforms $ε = 1.0$, which in turn outperforms $ε = 0.0$. 

### By examining the arm selection for each case, we notice that when $ε = 1.0$, it minics the random strategy as we already see in Example 1 and the cumulative average reward converges towards the average winning probability which is 0.52. 

### Instead, when $ε = 0.0$, there is no exploration, Arm 0 is always selected. This is because at the very beginning, the average reward for all arms are the same, they are initialized to 0. With <small>`arm = np.argmax(average_rewards)`</small>, we will select the index with the first occurence of the maximum value, namely Arm 0. Since Arm 0 has a winning probability of 0.37, the cumulative average reward also converges towards this value.

### This also explains why the performance of $ε = 1.0$ outperforms $ε = 0.0$. 

# Assignment questions

# <span style="color:blue"> [10 points] </span> Question 1: Epsilon decay algorithm

### With respect to the basic version of epsilon greedy strategy above, **epsilon decay** allows to adjust the value of epsilon during the learning process. More specifically, the epsilon value will decreases with time to allows more exploration at the beginning of the learning process when uncertainty is high, and then the focus is shifted towards exploitation.

### There are different variants of decaying algorithms. In the following, you need to implement a very simple version of decay: **the epsilon value decayed by 1% after each trail ($ε = ε*0.99$), and the epsilon value should not go below 0.05**.

## <span style="color:blue"> Q1.1: implement the epsilon decay algorithm by extending the basic version of epsilon greedy in Example 2

### Pay attention, in addition to rewards, selected arms, you should also return the **epsilon values used in each trail**, since it decays. This is needed for a future question. 

In [ ]:
### insert your code here 

### Provide a brief explanation to your code, focusing on the modification you made with respect to the basic version of epsilon greedy in Example 2.

----
<div style="color:blue">
    
**Answer**: insert your answer here

</div>

----

## <span style="color:blue">  Q1.2: Run an experiment with 1000 trails, at each trail the epsilon decay strategy is used. After that, compute also cumulative average rewards

### Pay attention, you need to run the experiment with the **initial epsilon set to 1.0** and the **decay rate set to 0.99**.

In [ ]:
### insert your code here 

## <span style="color:blue"> Q1.3:  Visualize the epsilon value, selected arm and the cumulative average reward  for each trail

### Pay attention, before plotting the selected arm and the cumulative average reward for each trail, you should **first plot the epsilon values used at each trail** to show it is decaying.

In [ ]:
### insert your code here 

### What kind of observation can be made based on the 3 plots? Add a brief discussion to the result.

----
<div style="color:blue">
    
**Answer**: insert your answer here

</div>

----

## <span style="color:blue"> Q1.4: Plot and compare the result (cumulative average reward) to the one obtained with the epsilon greedy strategy (without decay and $ε = 0.2$)

In [ ]:
### insert your code here

## <span style="color:blue"> Q1.5: What kind of observation can we make, if we compare the cumulative average reward obtained with Epsilon Decay (initial epsilon set to 1.0 and decay rate set to 0.99) against the one obtained with Epsilon Greedy with epsilon equals 1.0 (the one shown in Example 3).

----
<div style="color:blue">
    
**Answer**: insert your answer here

</div>

----

# <span style="color:blue"> [10 points] </span> Question 2: Optimistic Initial Values (OIV) strategy

<div style="text-align: left;">
  <img src="OIV.png" width="500">
</div>

## <span style="color:blue"> Q2.1: Implement the Optimistic Initial Values (OIV) algorithm

**Requirements**

Use the following function signature:

```python
def optimistic_initial_values_strategy(reward_probs, num_trials, optimistic_initial_value):

    return rewards, selected_arms
```

- `reward_probs`: winning probability of each arm. Use these probabilities only to simulate the reward after selecting an arm—not to identify the best arm directly.
- `num_trials`: total number of trials to run.
- `optimistic_initial_value`: initial estimated average reward assigned to every arm.

The function must return:
- `rewards`: the reward obtained in each trial.
- `selected_arms`: the index of the arm selected in each trial.

Return them in the same format as in Example 2 so that they can be used in the remaining questions.

> 💡 **Hint:** If needed, refer to the OIV pseudocode in the lecture slides.  
> Use it to understand the algorithmic steps, but adapt it to this exercise. The pseudocode will not run unchanged.

In [ ]:
### insert you code here

### Provide a brief explanation to your code, focusing on the difference with respect to the other strategies.

----
<div style="color:blue">
    
**Answer**: insert your answer here

</div>

----

## <span style="color:blue"> Q2.2: Explain how the initial values of the estimated average rewards for each arm will be set and why it is set in that way?

----
<div style="color:blue">
    
**Answer**: insert your answer here

</div>

----

## <span style="color:blue"> Q2.3: Run an experiment with 1000 trails, at each trail the Optimistic Initial Values (OIV) algorithm is used. After that, compute the cumulative average rewards.

In [ ]:
### insert you code here

## <span style="color:blue"> Q2.4:   Visualize the selected arm and the cumulative average reward 

In [ ]:
### insert you code here

### What kind of observation can you make based on the plot? Add a brief discussion of the result.

----
<div style="color:blue">
    
**Answer**: insert your answer here

</div>

----

## <span style="color:blue"> Q2.5: Plot and compare the result (cumulative average reward) obtained with OIV against the results obtained with random strategy, and the epsilon greedy algorithm (without decay and $ε = 0.2$)

In [ ]:
### insert your code here

### What kind of observation can you make based on the plot? Add a brief discussion of the result.

----
<div style="color:blue">
    
**Answer**: insert your answer here

</div>

----